<a href="https://colab.research.google.com/github/franz-hufana/Thesis-MobileNet/blob/main/Optimized_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Thesis System: MobileNetV2 and MobileNetV3 Based Model

In [7]:
import os
import gc
import zipfile
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from imageio.v2 import imread
from skimage.transform import resize
from tensorflow.keras import layers
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.applications import MobileNetV2, MobileNetV3Large
from tensorflow.keras import backend as K
from sklearn.utils import shuffle
from sklearn.metrics import confusion_matrix, f1_score, classification_report
from google.colab import files
from scipy.ndimage import label as cc_label
from matplotlib.patches import Rectangle

Import Necessary Datasets

In [ ]:
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
extract_path = "/content"

with zipfile.ZipFile(zip_name, 'r') as zip_ref:
  zip_ref.extractall(extract_path)

print(f'Dataset extracted: {zip_name}')

Labelling the datasets according to their Classes and number of classes

In [ ]:
folder_path = '/content/woods'

label_map = {
    'c' : 0, # For Cracks
    'f' : 1, # For Fuzz
    'k' : 2, # For knotholes
    'n' : 3 # For No Defects
}

class_name = ['cracks', 'fuzz', 'knothole', 'no defect']
num_classes = 4
img_size = (224, 224) # Normalizing the images' size

print(f"Name of Clases {class_name}\nFolder Path: {folder_path}")

1. Preprocessing the Datasets to fit the MobileNet requirements

2. Dataset Shuffling

3. Splitting Train and Validation Set for our model

In [ ]:
print('Step 1: Data Preprocessing....')
print('Locating Datasets')
image_paths = []
for roots, dirs, files_in_dir in os.walk(folder_path):
  for file_name in file_in_dir:
    print('\nChecking Datasets')
    if file_name.lower().endswith(('.png', '.jpg', '.jpeg')):
      image_paths.append(os.ath.join(root, file_name))

print('\nSorting Datasets according to label')
image_paths = sorted(image_paths)

valid_data = []
valid_labels = []
valid_names = []

for file_path in image_paths:
  file_name os.path.basename(file_path)
  prefix = file_name[0].lower()

  if prefix not in label_map:
    print(f"Skiping unknown file: {file_name}")
    continue

  try:
    im = imread(file_path)

    if im.ndim == 2:
      print('\nConverted GrayScale Images to 3-channel (RGB)')
      im = np.stack([im] *3, axis=-1)

    elif im.ndim == 3 and im.shpe[-1] == 4:
      print('\nConverted RGBA Images to RGB')
      im = im[:, :, :3]

    print('\nResizing images to 224x224')
    im = resize(im, img_size, anti_aliasing = True, preserve_range = True)
    im = im.astype(np.float32)
    im = (im/127.5) - 1.0

    valid_data.append(im)
    valid_labels.append(label_map[prefix])
    valid_names.append(file_name)

  except Exception as e:
    print(f"Error reading {file_name}: {e}")


data = np.array(valid_data, dtype=np.float32)
labels = np.array(valid_labels, dtype=np.int32)

print(f'\nData Information (number of Datasets, Height, Width, Color Code): {data.shape}')
print(f'Label Information (number of Datasets): {labels.shape}')

print('\nSample Loaded files:')
for i in range(min(10, len(valid_names))):
  print(f"{valid_names[i]} --->> {class_name[labels[i]]}")


print('\nStep 2: Dataset Shuffling')
data, labels = shuffle(data, labels, random_state=42)
print('\nDataset Shuffled once only.')

print('\nStep 3: Splitting Dataset for Train and Validation....')

# For Validation
val_fraction = 0.2
val_size = int(len(data) * val_fraction)


x_val = data[:val_size]
y_val = labels[:val_size]

x_train = data[val_size:]
y_train = labels[val_size:]

print(f'\nTraining Set: {y_train.shape}\nValidation Set: {y_val.shape}')


Creating Layers for Data Augmentations

In [3]:
def get_data_augmentation():
  return tf.keras.Sequential([
      layers.RandomFlip("horizontal"),
      layers.RandomRotation(0.22),
      layers.RandomZoom(0.2),
      layers.RandomContrast(0.2),
  ], name='data_augmentation')

Load the Models and Create New Classifier Head for Specific Task

In [4]:
def build_model(backbone_name):
  inputs = layers.Input(Shape=(224, 224, 3), name="input_image")
  data_augmentation = get_data_augmentation()

  # Section for the Model
  if backbone_name == "MobileNetV2":
    base_model = MobileNetV2(
        input_shape=(224, 224, 3),
        include_top=False,
        weights="imagenet",
    )
  elif backbone_name == "MobileNetV3":
    base_model = MobileNetV3Large(
        input_shape=(224, 224, 3),
        include_top=False,
        weights="imagenet",
        include_preprocessing=False
    )
  else:
    raise ValueError("Unsupported Model")

# Exclude already trained layers
  base_model.trainable = False

# New Classifier Head for the Model
  x = data_augmentation(inputs)
  x = base_model(x, training=False)
  x = GlobalAveragePooling2D(name="gap")(x)
  x = layers.Dropout(0.4, name="dropout")(x)
  outputs = Dense(num_classes, activation = "softmax", name="wood_output")(x)

  model = Model(inputs=inputs, outputs=outputs, name=f"{backbone_name}_wood_model")
  model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
  )

  return model

Setting Up the Model for Training and Evaluation

In [ ]:
def train_and_evaluate(backbone_name, epochs=20, bathc_size=32):

  print("\n" + "=" * 60)
  print(f"TRAINING {backbone_name}")
  print("=" * 60)

  k.clear_session()
  gc.collection()
  model = build_model(backbone_name)

# Model Summary
  print(f"\nMODEL SUMMARY:"{backbone_name})
  model.summary()

# Training Parameters
  history = model.fit(
      x_train,
      y_train,
      validation_data(X_val, y_val),
      batch_size=batch_size,
      epochs=epochs,
      verbose=2,
      shuffle=True
  )

# Prediction on validation set
  val_probs = model.predict(X_val, batch_size=batch_size, verbose=0)
  val_preds = np.argmax(val_probs, axis=1)

# F1 score
  print(f'F1-score of {backbone_name}')
  f1_macro = f1_score(y_val, val_preds, average='macro')
  f1_micro = f1_score(y_val, val_preds, average='micro')
  f1_weighted = f1_score(y_val, val_preds, average='weighted')
  print(f"\nMacro F1   : {f1_macro:.4f}")
  print(f"Micro F1   : {f1_micro:.4f}")
  print(f"Weighted F1: {f1_weighted:.4f}")

# Summary of Classification
  print(f"\nClassification Report of {backbone_name}")
  print(classification_report(y_val, val_preds, target_names=class_name))

# For Confusion Matrix
  cm = confusion_matrix(y_val, val_preds)

  metrics_dict = {
    "macro_f1": f1_macro,
    "micro_f1": f1_micro,
    "weighted_f1": f1_weighted,
    "val_accuracy_last": history.history["val_accuracy"][-1],
    "val_loss_last": history.history["val_loss"][-1]
  }

  return model, history, cm, metric dict

Training the Models Using the New Classifier Head

In [ ]:
mobilenetv2_model, history_v2, cm_v2, metrics_v2 = train_and_evaluate(
    backbone_name="MobileNetV2",
    epochs=20,
    batch_size=32
)

mobilenetv3_model, history_v3, cm_v3, metrics_v3 = train_and_evaluate(
    backbone_name="MobileNetV3Large",
    epochs=20,
    batch_size=32
)

Here is the Section of the Learning Curve for both Models

In [ ]:
# For MobileNetV2
plt.figure(figsize=(10, 5))
plt.plot(history_v2.history['accuracy'], label='MobileNetV2 Train Accuracy')
plt.plot(history_v2.history['val_accuracy'], label='MobileNetV2 Val Accuracy')
plt.plot(history_v2.history['loss'], label='MobileNetV2 Train Loss')
plt.plot(history_v2.history['val_loss'], label='MobileNetV2 Val Loss')
plt.title('MobileNetV2 Training and Validation Performance')
plt.xlabel('Epoch')
plt.ylabel('Accuracy / Loss')
plt.legend()
plt.grid(True)
plt.show()

# For MobileNetV3
plt.figure(figsize=(10, 5))
plt.plot(history_v3.history['accuracy'], label='MobileNetV3 Train Accuracy')
plt.plot(history_v3.history['val_accuracy'], label='MobileNetV3 Val Accuracy')
plt.plot(history_v3.history['loss'], label='MobileNetV3 Train Loss')
plt.plot(history_v3.history['val_loss'], label='MobileNetV3 Val Loss')
plt.title('MobileNetV3Large Training and Validation Performance')
plt.xlabel('Epoch')
plt.ylabel('Accuracy / Loss')
plt.legend()
plt.grid(True)
plt.show()

Confusion Matrix of Each Model

In [ ]:
# For MobileNetV2
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm_v2,
    annot=True,
    fmt='d',
    xticklabels=class_names,
    yticklabels=class_names,
    cmap='Blues'
)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('MobileNetV2 Confusion Matrix')
plt.show()

# For MobileNetV3
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm_v3,
    annot=True,
    fmt='d',
    xticklabels=class_names,
    yticklabels=class_names,
    cmap='Blues'
)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('MobileNetV3Large Confusion Matrix')
plt.show()

Comparative Results of Both Models


In [ ]:
print("MobileNetV2")
for k, v in metrics_v2.items():
    print(f"{k}: {v:.4f}")

print("\nMobileNetV3Large")
for k, v in metrics_v3.items():
    print(f"{k}: {v:.4f}")

Using Bound Box for our Model to simulate real-time use

In [ ]:
# =========================================================
# 16. PATCH PREPROCESSING FOR SLIDING-WINDOW SCANNING
# =========================================================
def preprocess_patch(patch, target_size=(224, 224)):
    """
    Preprocess one patch the same way as training images.
    """
    patch_resized = resize(
        patch,
        target_size,
        anti_aliasing=True,
        preserve_range=True
    ).astype(np.float32)

    # same manual preprocessing used in training
    patch_resized = (patch_resized / 127.5) - 1.0
    return patch_resized


# =========================================================
# 17. SLIDING-WINDOW MULTI-DEFECT SCAN
# =========================================================
def sliding_window_scan(
    image_array,
    model,
    class_names,
    window_size=224,
    step=112,
    conf_threshold=0.60
):
    """
    Scan a large wood image patch-by-patch.
    Each patch is classified individually.

    Returns:
    - detections: positive defect patches
    - percentages: approximate percentage of each defect
    - patch_records: all scanned patch predictions
    - total_patches: total number of patches scanned
    """

    h, w = image_array.shape[:2]
    detections = []
    patch_records = []

    total_patches = 0
    class_patch_count = {c: 0 for c in class_names}

    for y in range(0, h - window_size + 1, step):
        for x in range(0, w - window_size + 1, step):
            patch = image_array[y:y + window_size, x:x + window_size]

            if patch.shape[0] != window_size or patch.shape[1] != window_size:
                continue

            patch_input = preprocess_patch(patch)
            patch_input = np.expand_dims(patch_input, axis=0)

            probs = model.predict(patch_input, verbose=0)[0]
            pred_idx = np.argmax(probs)
            pred_class = class_names[pred_idx]
            confidence = float(probs[pred_idx])

            total_patches += 1

            patch_records.append({
                "x": x,
                "y": y,
                "w": window_size,
                "h": window_size,
                "pred_class": pred_class,
                "confidence": confidence,
                "probs": probs
            })

            if pred_class != "no defect" and confidence >= conf_threshold:
                detections.append({
                    "x1": x,
                    "y1": y,
                    "x2": x + window_size,
                    "y2": y + window_size,
                    "class": pred_class,
                    "confidence": confidence
                })
                class_patch_count[pred_class] += 1

    percentages = {}
    defect_patch_total = sum(
        class_patch_count[c] for c in class_names if c != "no defect"
    )

    if defect_patch_total == 0:
        for c in class_names:
            if c != "no defect":
                percentages[c] = 0.0
    else:
        for c in class_names:
            if c != "no defect":
                percentages[c] = (class_patch_count[c] / defect_patch_total) * 100.0

    return detections, percentages, patch_records, total_patches


# =========================================================
# 18. MERGE PATCH DETECTIONS INTO APPROXIMATE BOXES
# =========================================================
def detections_to_boxes(
    image_shape,
    detections,
    target_class,
    step=112,
    min_region_size=1
):
    """
    Merge nearby positive patches into larger approximate
    bounding boxes using connected components.
    """

    h, w = image_shape[:2]

    if len(detections) == 0:
        return []

    grid_h = int(np.ceil(h / step))
    grid_w = int(np.ceil(w / step))
    mask = np.zeros((grid_h, grid_w), dtype=np.uint8)

    for det in detections:
        if det["class"] != target_class:
            continue

        gx1 = det["x1"] // step
        gy1 = det["y1"] // step
        gx2 = max(gx1, (det["x2"] - 1) // step)
        gy2 = max(gy1, (det["y2"] - 1) // step)

        mask[gy1:gy2 + 1, gx1:gx2 + 1] = 1

    labeled, num = cc_label(mask)
    boxes = []

    for region_id in range(1, num + 1):
        ys, xs = np.where(labeled == region_id)

        if len(xs) < min_region_size:
            continue

        x1 = int(xs.min() * step)
        y1 = int(ys.min() * step)
        x2 = int(min(w, (xs.max() + 1) * step))
        y2 = int(min(h, (ys.max() + 1) * step))

        boxes.append((x1, y1, x2, y2))

    return boxes


# =========================================================
# 19. PREDICT MULTIPLE DEFECTS IN ONE IMAGE
# =========================================================
def predict_multiple_defects_in_image(
    image_path,
    model,
    class_names,
    window_size=224,
    step=112,
    conf_threshold=0.60
):
    """
    Reads one large image, scans it patch-by-patch,
    and returns approximate multi-defect localization.
    """

    img = imread(image_path)

    if img.ndim == 2:
        img = np.stack([img] * 3, axis=-1)
    elif img.ndim == 3 and img.shape[-1] == 4:
        img = img[:, :, :3]

    img = img.astype(np.float32)

    detections, percentages, patch_records, total_patches = sliding_window_scan(
        image_array=img,
        model=model,
        class_names=class_names,
        window_size=window_size,
        step=step,
        conf_threshold=conf_threshold
    )

    final_boxes = {}
    for cls in class_names:
        if cls == "no defect":
            continue

        final_boxes[cls] = detections_to_boxes(
            image_shape=img.shape,
            detections=detections,
            target_class=cls,
            step=step
        )

    region_counts = {cls: len(final_boxes[cls]) for cls in final_boxes}

    return img.astype(np.uint8), detections, percentages, final_boxes, region_counts, patch_records, total_patches


# =========================================================
# 20. VISUALIZE DETECTED DEFECTS
# =========================================================
def show_detected_defects(image, final_boxes, percentages, region_counts, model_name):
    """
    Draw approximate bounding boxes for each detected defect type.
    """
    color_map = {
        "cracks": "red",
        "fuzz": "orange",
        "knothole": "blue"
    }

    fig, ax = plt.subplots(figsize=(12, 8))
    ax.imshow(image.astype(np.uint8))

    for defect_class, boxes in final_boxes.items():
        color = color_map.get(defect_class, "yellow")

        for (x1, y1, x2, y2) in boxes:
            rect = Rectangle(
                (x1, y1),
                x2 - x1,
                y2 - y1,
                linewidth=2,
                edgecolor=color,
                facecolor='none'
            )
            ax.add_patch(rect)

            label_text = (
                f"{defect_class} | "
                f"count={region_counts[defect_class]} | "
                f"{percentages.get(defect_class, 0):.1f}%"
            )

            ax.text(
                x1,
                max(10, y1 - 5),
                label_text,
                color="white",
                fontsize=10,
                bbox=dict(facecolor=color, alpha=0.7)
            )

    ax.set_title(f"Approximate Multiple Surface Defect Detection - {model_name}")
    ax.axis("off")
    plt.show()


# =========================================================
# 21. SEPARATE TESTING FUNCTION FOR COMPARISON
# =========================================================
def test_model_on_large_image(
    model,
    model_name,
    image_path,
    class_names,
    window_size=224,
    step=112,
    conf_threshold=0.60
):
    """
    Test one trained model on the same large image.
    This is separated so MobileNetV2 and MobileNetV3Large
    can be compared clearly.
    """

    print("\n" + "=" * 70)
    print(f"TESTING {model_name} ON LARGE IMAGE")
    print("=" * 70)

    image_result, detections, percentages, final_boxes, region_counts, patch_records, total_patches = predict_multiple_defects_in_image(
        image_path=image_path,
        model=model,
        class_names=class_names,
        window_size=window_size,
        step=step,
        conf_threshold=conf_threshold
    )

    print(f"Model used: {model_name}")
    print(f"Test image: {image_path}")
    print(f"Total scanned patches: {total_patches}")
    print(f"Positive defect patches: {len(detections)}")

    print("\nApproximate Percentage of Each Defect")
    for defect, pct in percentages.items():
        print(f"{defect}: {pct:.2f}%")

    print("\nApproximate Number of Defect Regions")
    for defect, cnt in region_counts.items():
        print(f"{defect}: {cnt}")

    show_detected_defects(
        image=image_result,
        final_boxes=final_boxes,
        percentages=percentages,
        region_counts=region_counts,
        model_name=model_name
    )

    return {
        "model_name": model_name,
        "total_patches": total_patches,
        "positive_patches": len(detections),
        "percentages": percentages,
        "region_counts": region_counts
    }


# =========================================================
# 22. UPLOAD ONE TEST IMAGE FOR COMPARATIVE SCANNING
# =========================================================
print("\n=========================================================")
print("STEP 5: UPLOAD ONE LARGE WOOD IMAGE FOR COMPARATIVE TESTING")
print("=========================================================")

uploaded_test = files.upload()
test_image_name = list(uploaded_test.keys())[0]


# =========================================================
# 23. TEST MOBILENETV2 SEPARATELY
# =========================================================
results_v2_scan = test_model_on_large_image(
    model=mobilenetv2_model,
    model_name="MobileNetV2",
    image_path=test_image_name,
    class_names=class_names,
    window_size=224,
    step=112,
    conf_threshold=0.60
)


# =========================================================
# 24. TEST MOBILENETV3LARGE SEPARATELY
# =========================================================
results_v3_scan = test_model_on_large_image(
    model=mobilenetv3_model,
    model_name="MobileNetV3Large",
    image_path=test_image_name,
    class_names=class_names,
    window_size=224,
    step=112,
    conf_threshold=0.60
)


# =========================================================
# 25. SIMPLE COMPARATIVE SUMMARY
# =========================================================
print("\n" + "=" * 70)
print("COMPARATIVE SCANNING SUMMARY")
print("=" * 70)

print("\nMobileNetV2")
print("Positive defect patches:", results_v2_scan["positive_patches"])
for defect in results_v2_scan["percentages"]:
    print(f"{defect}: {results_v2_scan['percentages'][defect]:.2f}% | count={results_v2_scan['region_counts'][defect]}")

print("\nMobileNetV3Large")
print("Positive defect patches:", results_v3_scan["positive_patches"])
for defect in results_v3_scan["percentages"]:
    print(f"{defect}: {results_v3_scan['percentages'][defect]:.2f}% | count={results_v3_scan['region_counts'][defect]}")


# =========================================================
# 26. OPTIONAL: SAVE THE TRAINED MODELS
# =========================================================
mobilenetv2_model.save("wood_defect_mobilenetv2_system.h5")
mobilenetv3_model.save("wood_defect_mobilenetv3large_system.h5")

print("\nModels saved successfully:")
print("- wood_defect_mobilenetv2_system.h5")
print("- wood_defect_mobilenetv3large_system.h5")


# =========================================================
# 27. IMPORTANT NOTE FOR PANEL / THESIS
# =========================================================
print("\n=========================================================")
print("IMPORTANT NOTE")
print("=========================================================")
print("This multi-defect scanning feature is classification-based.")
print("It scans one image patch-by-patch and groups positive patches.")
print("Therefore:")
print("- percentages are approximate")
print("- counts are approximate")
print("- bounding boxes are approximate")
print("For exact bounding boxes, a true object detection dataset is needed.")